# 04 — Modelling

Logistic Regression baseline + XGBoost with proper imbalance handling.
Honest train/test split that respects temporal drift findings from 01.

In [ ]:
# Cell 1 — setup
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score
from fraud.data_loader import load_raw
from fraud.models import temporal_split, feature_columns, build_baseline, build_xgb

# Cell 2 — load + temporal split
# load_raw() resolves the path from the package location (CWD-independent)
# and validates the data contract: 284,807 rows, 492 frauds, no nulls.
df = load_raw()
train, test = temporal_split(df, train_fraction=0.80)
feats = feature_columns(df)
X_train, y_train = train[feats], train["Class"]
X_test,  y_test  = test[feats],  test["Class"]
print(f"train {len(train):,} (fraud {int(y_train.sum())}) | test {len(test):,} (fraud {int(y_test.sum())})")

# Cell 3 — baseline
base = build_baseline().fit(X_train, y_train)
p_base = base.predict_proba(X_test)[:, 1]
print(f"LogReg   PR-AUC {average_precision_score(y_test, p_base):.3f}  ROC-AUC {roc_auc_score(y_test, p_base):.3f}")

# Cell 4 — XGBoost
xgb = build_xgb(y_train).fit(X_train, y_train)
p_xgb = xgb.predict_proba(X_test)[:, 1]
print(f"XGBoost  PR-AUC {average_precision_score(y_test, p_xgb):.3f}  ROC-AUC {roc_auc_score(y_test, p_xgb):.3f}")